# ComfyUI + Wan 2.2 TI2V-5B — Kaggle
Workflow: Wan 2.2 TI2V-5B (Texto-a-Video + Imagen-a-Video).
Usa Cloudflare Tunnel (no necesita token ni cuenta).Modelo ligero (~16.9 GB) que cabe en el límite de 20 GB de Kaggle.

In [ ]:
import os, shutil, subprocess, time, requests

BASE = os.getcwd()
COMFY = os.path.join(BASE, "ComfyUI")
print("Directorio base:", BASE)
print("ComfyUI en:", COMFY)

# Verificar espacio disponible
import shutil as sh
total, used, free = sh.disk_usage(BASE)
print(f"  Espacio: total={total//2**30}GB usado={used//2**30}GB libre={free//2**30}GB")

if os.path.exists(COMFY):
    shutil.rmtree(COMFY)

!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
os.chdir(COMFY)

!pip install -q -r requirements.txt

# Crear directorios de modelos
os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/loras", exist_ok=True)
os.makedirs("input", exist_ok=True)

print("Descargando modelos...")
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download

def download_hf(repo, path, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    name = os.path.basename(path)
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 10**6:
        print(f"  ✓ {name} ya existe ({os.path.getsize(dest)//2**30} GB), omitir")
        return
    print(f"  Descargando {name}...")
    try:
        hf_hub_download(repo_id=repo, filename=path, local_dir=dest_dir, local_dir_use_symlinks=False)
        size = os.path.getsize(dest) if os.path.exists(dest) else 0
        print(f"    ✓ {name}: {size//2**30} GB")
    except Exception as e:
        print(f"    ✗ Error descargando {name}: {e}")
        raise

# 1. Diffusion model (9.31 GB)
download_hf("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", "models/diffusion_models")

# 2. VAE (1.31 GB)
download_hf("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan2.2_vae.safetensors", "models/vae")

# 3. Text encoder (6.27 GB)
download_hf("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "models/text_encoders")

# Verificar tamaños
print("\n=== Modelos descargados ===\n")
total_size = 0
for root, dirs, files in os.walk("models"):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        total_size += size
        print(f"  {f}: {size//2**30} GB ({size//2**20} MB)")
print(f"\nTotal: {total_size//2**30} GB")

# Limpiar caché para dejar espacio
!rm -rf /root/.cache/pip /tmp/*
!find . -type d -name "__pycache__" -exec rm -rf {} + 2>/dev/null

print("\nModelos listos. Iniciando ComfyUI...")

log = open("comfyui.log", "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],    cwd=COMFY, stdout=log, stderr=log
)
print("ComfyUI PID:", proc.pid)

for i in range(120):
    try:
        r = requests.get("http://localhost:8188", timeout=5)
        if r.status_code == 200:
            print("ComfyUI listo en :8188")
            break
    except:
        pass
    time.sleep(1)
else:
    print("ComfyUI no arranco. Revisa comfyui.log:")
    print(open("comfyui.log").read()[-1500:])

In [ ]:
print("Instalando Cloudflare Tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

cf_log = open("cloudflare.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=cf_log, stderr=cf_log
)
time.sleep(8)

log_content = open("cloudflare.log").read()
import re
match = re.search(r'https://[a-z-]+\.trycloudflare\.com', log_content)
if match:
    url = match.group()
    print("=" * 60)
    print(f"ComfyUI en linea!")
    print(f"Enlace: {url}")
    print("=" * 60)
else:
    print("No se encontro la URL. Revisa cloudflare.log:")
    print(log_content[-1500:])

## Workflow: Wan 2.2 TI2V-5B
- Carga el flujo de trabajo TI2V-5B (busca 'Wan 2.2 5B' en Templates o arrastra el JSON)
- Es texto-a-video e imagen-a-video con un solo modelo

## Modelos descargados
| Tipo | Archivo | Tamaño |
|------|---------|--------|
| Diffusion | `wan2.2_ti2v_5B_fp16.safetensors` | 9.31 GB |
| Text encoder | `umt5_xxl_fp8_e4m3fn_scaled.safetensors` | 6.27 GB |
| VAE | `wan2.2_vae.safetensors` | 1.31 GB |
| **Total** | | **~16.9 GB** |

## Uso
1. Elige **Texto-a-Video** o **Imagen-a-Video** en el workflow
2. Para I2V: sube una imagen en el nodo LoadImage
3. Escribe el prompt (movimiento, cámara, estilo)
4. Ajusta resolución (1280x704 recomendado) y duración
5. Pulsa **Queue**

## Notas
- Cierra el tunnel en Kaggle: `File` -> `Stop execution`
- GPU: T4 x2 (32 GB VRAM) - ComfyUI usa una GPU automáticamente- Los modelos se borran al cerrar la sesión